In [ ]:
!pip install -U unsloth trl datasets accelerate bitsandbytes sentencepiece jsonschema peft scikit-learn evaluate absl-py rouge-score


In [ ]:
!pip install --upgrade "torchvision>=0.25.0"
!pip install datasets==4.3.0
import unsloth  # ← must be first
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import csv
import json
from pathlib import Path

# === Input paths ===
CSV_PATH = Path("top_1000_task_messages.csv")
LABELS_PATH = Path("jira_tasks_first_1000_final.json")  # list of JSON dicts
OUTPUT_PATH = Path("train.jsonl")

# If your labels use a different field for matching, change this:
LABEL_MATCH_KEY = "issue_creation_date"   # e.g., "timestamp" if that's what your labels use

# === Step 1. Load labels and index by timestamp-like key ===
with LABELS_PATH.open("r", encoding="utf-8") as f:
    labels = json.load(f)

label_lookup = {}
for item in labels:
    key = item.get(LABEL_MATCH_KEY)
    if key is not None:
        label_lookup[str(key)] = item  # stringify to match CSV string timestamps

# === Step 2. Read Slack CSV and pair with labels ===
jsonl_records = []
with CSV_PATH.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        ts = str(row.get("timestamp", "")).strip()
        slack_text = (row.get("cleaned_text") or "").strip()
        if not ts or not slack_text:
            continue

        label = label_lookup.get(ts)
        if not label:
            continue  # skip if no matching label

        record = {
            "input": {
                "timestamp": ts,
                "text": slack_text
            },
            "output": label  # keep as JSON object (not a pretty-printed string)
        }
        jsonl_records.append(record)

# === Step 3. Write JSONL ===
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for rec in jsonl_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f" Saved {len(jsonl_records)} examples to {OUTPUT_PATH}")

In [ ]:
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
login(token=hf_token)

In [ ]:
# === Step 4. Train/Val split ===
from sklearn.model_selection import train_test_split

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    jsonl_records = [json.loads(line) for line in f]

train_records, val_records = train_test_split(jsonl_records, test_size=0.2, random_state=42)

for filename, records in [("train.jsonl", train_records), ("val.jsonl", val_records)]:
    with open(filename, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Train: {len(train_records)} | Val: {len(val_records)}")


In [ ]:
# =============================================================
# === 2. MODEL LOADING WITH UNSLOTH ===
# =============================================================
import torch
import re
import json
import datetime as dt
import evaluate
import pandas as pd
from tqdm import tqdm
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

MODEL_NAME = "unsloth/Qwen2.5-14B-Instruct"
MAX_SEQ_LENGTH = 2048

model, tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,          # QLoRA 4-bit
    dtype=None,                 # Auto-detect (bfloat16 on Ampere+)
)

# Apply LoRA via Unsloth (faster than vanilla PEFT)
# Apply LoRA via Unsloth (faster than vanilla PEFT)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,              # ← change from 0.05 to 0 for full Unsloth speed
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    # task_type="CAUSAL_LM",    # ← remove this line, Unsloth handles it internally
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


In [ ]:
import subprocess
result = subprocess.run(["which", "gcc"], capture_output=True, text=True)
print("gcc path:", result.stdout)

# Install if missing
!apt-get update -qq && apt-get install -y gcc g++ build-essential

In [ ]:
!gcc --version

In [ ]:
import os
os.environ["CC"] = "gcc"
os.environ["CXX"] = "g++"

In [ ]:
# =============================================================
# === 3. SYSTEM PROMPT & INFERENCE HELPERS ===
# =============================================================
SYSTEM = """
"Extract a Jira ticket JSON from a Slack message. If no clear action, return a JSON object with a field no_action set to true. Output JSON only."

"""

def _last_balanced_json(text: str):
    """Extract the last complete balanced JSON object from text."""
    start, depth, spans = None, 0, []
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    spans.append((start, i + 1))
                    start = None
    return text[spans[-1][0]:spans[-1][1]] if spans else None

def _project_to_schema(obj, ts_fallback):
    if not isinstance(obj, dict) or obj.get("no_action") is True:
        return {"task_summary": "null", "assignee": None, "issue_creation_date": ts_fallback}
    return {
        "task_summary": obj.get("task_summary") if obj.get("task_summary") else "null",
        "assignee": obj.get("assignee"),
        "issue_creation_date": obj.get("issue_creation_date", ts_fallback)
    }

def predict_internal(model_obj, slack_input):
    ts = slack_input.get("timestamp") if isinstance(slack_input, dict) else None
    user_str = f"[{ts}] {slack_input.get('text')}" if ts else str(slack_input)

    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_str}
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model_obj.device)

    # Use Unsloth's fast inference mode
    FastLanguageModel.for_inference(model_obj)
    with torch.no_grad():
        out = model_obj.generate(
            **ids,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tok.eos_token_id
        )

    gen_text = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    obj_cand = _last_balanced_json(gen_text)
    try:
        parsed = json.loads(obj_cand)
        return _project_to_schema(parsed, ts)
    except Exception:
        return {"task_summary": "null", "assignee": None, "issue_creation_date": ts}

# Quick sanity check on base model
print(predict_internal(model, {"timestamp": "2024-01-01", "text": "<@Ketki> Can you make the discussed changes in the final project slides?"}))


In [ ]:
# =============================================================
# === 4. DATASET PREPARATION FOR SFTTRAINER ===
# =============================================================
ds = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})

def map_to_text(ex):
    user_content = f"[{ex['input'].get('timestamp')}] {ex['input'].get('text')}"
    out_json = json.dumps(ex["output"], ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    convo = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": out_json}
    ]
    return {"text": tok.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)}

ds = ds.map(map_to_text, remove_columns=ds["train"].column_names)


In [ ]:
# =============================================================
# === 5. TRAINING ===
# =============================================================
# Switch back to training mode before SFTTrainer
FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    processing_class=tok,          # ← add this line
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        output_dir="./slack2jira-results",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=2e-4,
        bf16=True,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        load_best_model_at_end=True,
        report_to="none",
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
    ),
)

In [ ]:
# =============================================================
# === 6. ENHANCED EVALUATION SUITE ===
# =============================================================
# Metrics inspired by the "Beyond BLEU and ROUGE" framework:
#   - ROUGE (baseline / traditional metric for comparison)
#   - BERTScore (semantic coherence of task_summary — slide 14)
#   - Field-Level Exact Match (factuality at the atomic-fact level — slide 9-12)
#   - Assignee Accuracy (structured output correctness)
#   - Date Accuracy (structured output correctness)
#   - JSON Validity Rate (structural reliability)
#   - No-Action Detection F1 (task-specific binary classification)
# =============================================================

rouge_metric = evaluate.load("rouge")

# BERTScore requires bert_score package (installed above)
from bert_score import score as bert_score_fn

# ------------------------------------------------------------------
# Helper: collect predictions & references from a model
# ------------------------------------------------------------------
def collect_predictions(model_obj, records, n=50):
    """
    Returns two parallel lists:
      preds : list of projected-schema dicts (model output)
      refs  : list of projected-schema dicts (ground truth)
    """
    FastLanguageModel.for_inference(model_obj)
    preds, refs = [], []
    for ex in tqdm(records[:n], desc="Generating"):
        pred = predict_internal(model_obj, ex["input"])
        ref  = _project_to_schema(ex["output"], ex["input"].get("timestamp"))
        preds.append(pred)
        refs.append(ref)
    return preds, refs

# ------------------------------------------------------------------
# Metric 1 – ROUGE  (traditional baseline, slide 6)
# Measures n-gram overlap on the full JSON string.
# Limitation: penalises synonym use and brief-but-correct summaries.
# ------------------------------------------------------------------
def metric_rouge(preds, refs):
    pred_strs = [json.dumps(p, sort_keys=True) for p in preds]
    ref_strs  = [json.dumps(r, sort_keys=True) for r in refs]
    return rouge_metric.compute(predictions=pred_strs, references=ref_strs)

# ------------------------------------------------------------------
# Metric 2 – BERTScore on task_summary  (semantic coherence, slide 14)
# Uses contextual BERT embeddings → cosine similarity.
# Captures synonyms & paraphrases that ROUGE misses.
# ------------------------------------------------------------------
def metric_bertscore(preds, refs):
    pred_summaries = [str(p.get("task_summary") or "") for p in preds]
    ref_summaries  = [str(r.get("task_summary") or "") for r in refs]
    P, R, F1 = bert_score_fn(pred_summaries, ref_summaries,
                              lang="en", verbose=False)
    return {
        "bertscore_precision": float(P.mean()),
        "bertscore_recall":    float(R.mean()),
        "bertscore_f1":        float(F1.mean()),
    }

# ------------------------------------------------------------------
# Metric 3 – Field-Level Exact Match  (atomic-fact accuracy, slide 9-12)
# FactScore-style: each structured field = one "atomic fact".
# Checks whether the model correctly extracted each field independently.
# ------------------------------------------------------------------
def metric_field_exact_match(preds, refs):
    fields = ["task_summary", "assignee", "issue_creation_date"]
    results = {}
    for field in fields:
        correct = sum(
            1 for p, r in zip(preds, refs)
            if str(p.get(field) or "").strip().lower() ==
               str(r.get(field) or "").strip().lower()
        )
        results[f"exact_match_{field}"] = correct / len(preds) if preds else 0.0
    # Overall: all three fields correct simultaneously
    all_correct = sum(
        1 for p, r in zip(preds, refs)
        if all(
            str(p.get(f) or "").strip().lower() ==
            str(r.get(f) or "").strip().lower()
            for f in fields
        )
    )
    results["exact_match_all_fields"] = all_correct / len(preds) if preds else 0.0
    return results

# ------------------------------------------------------------------
# Metric 4 – Assignee Accuracy
# Binary classification metric: did the model identify the right person?
# Critical for Jira ticket routing.
# ------------------------------------------------------------------
def metric_assignee_accuracy(preds, refs):
    correct = sum(
        1 for p, r in zip(preds, refs)
        if str(p.get("assignee") or "").strip().lower() ==
           str(r.get("assignee") or "").strip().lower()
    )
    return {"assignee_accuracy": correct / len(preds) if preds else 0.0}

# ------------------------------------------------------------------
# Metric 5 – Date Accuracy
# Checks ISO date extraction quality (exact string match on date portion).
# ------------------------------------------------------------------
def _extract_date(ts):
    """Return YYYY-MM-DD portion of a timestamp string, or the full string."""
    if ts is None:
        return ""
    s = str(ts).strip()
    # Accept YYYY-MM-DD at the start
    m = re.match(r"(\d{4}-\d{2}-\d{2})", s)
    return m.group(1) if m else s.lower()

def metric_date_accuracy(preds, refs):
    correct = sum(
        1 for p, r in zip(preds, refs)
        if _extract_date(p.get("issue_creation_date")) ==
           _extract_date(r.get("issue_creation_date"))
    )
    return {"date_accuracy": correct / len(preds) if preds else 0.0}

# ------------------------------------------------------------------
# Metric 6 – JSON Validity Rate  (structural reliability, slide 27-29)
# Measures what fraction of model outputs are parseable JSON.
# Low validity = model failing at the most basic output requirement.
# ------------------------------------------------------------------
def metric_json_validity(model_obj, records, n=50):
    FastLanguageModel.for_inference(model_obj)
    valid = 0
    for ex in tqdm(records[:n], desc="JSON validity"):
        ts = ex["input"].get("timestamp")
        user_str = f"[{ts}] {ex['input'].get('text')}"
        messages = [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": user_str}
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        ids = tok(prompt, return_tensors="pt").to(model_obj.device)
        with torch.no_grad():
            out = model_obj.generate(
                **ids, max_new_tokens=256, do_sample=False,
                pad_token_id=tok.eos_token_id
            )
        gen_text = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
        obj_cand = _last_balanced_json(gen_text)
        try:
            if obj_cand:
                json.loads(obj_cand)
                valid += 1
        except Exception:
            pass
    return {"json_validity_rate": valid / n if n > 0 else 0.0}

# ------------------------------------------------------------------
# Metric 7 – No-Action Detection F1  (task-specific classification)
# Many Slack messages require no Jira ticket. Detecting these correctly
# avoids polluting the backlog with spurious tickets.
# Precision/Recall/F1 modelled as in binary classification.
# ------------------------------------------------------------------
def metric_no_action_f1(preds, refs):
    def is_no_action(obj):
        return (obj.get("task_summary") in (None, "null", "") and
                obj.get("assignee") is None)

    tp = sum(1 for p, r in zip(preds, refs) if is_no_action(p) and is_no_action(r))
    fp = sum(1 for p, r in zip(preds, refs) if is_no_action(p) and not is_no_action(r))
    fn = sum(1 for p, r in zip(preds, refs) if not is_no_action(p) and is_no_action(r))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return {
        "no_action_precision": precision,
        "no_action_recall":    recall,
        "no_action_f1":        f1,
    }


In [ ]:
# ------------------------------------------------------------------
# Master evaluation function: runs all metrics and returns a flat dict
# ------------------------------------------------------------------
def evaluate_model(model_obj, records, n=50, label=""):
    print(f"\n{'='*50}")
    print(f"  Evaluating: {label}")
    print(f"{'='*50}")

    preds, refs = collect_predictions(model_obj, records, n=n)

    scores = {}
    print("   ROUGE...")
    scores.update(metric_rouge(preds, refs))

    print("   BERTScore (task_summary)...")
    scores.update(metric_bertscore(preds, refs))

    print("   Field-level exact match...")
    scores.update(metric_field_exact_match(preds, refs))

    print("  Assignee accuracy...")
    scores.update(metric_assignee_accuracy(preds, refs))

    print("  Date accuracy...")
    scores.update(metric_date_accuracy(preds, refs))

    print("  No-action detection F1...")
    scores.update(metric_no_action_f1(preds, refs))

    # JSON validity needs raw model output (re-run inference)
    print("  JSON validity rate...")
    scores.update(metric_json_validity(model_obj, records, n=n))

    return scores


In [ ]:
# =============================================================
# === 7. RUN EVALUATIONS ===
# =============================================================
EVAL_N = 50  # number of validation examples to evaluate

print("\n--- Evaluating Base Model ---")
base_metrics = evaluate_model(model, val_records, n=EVAL_N, label="Base Model (Qwen2.5-14B)")

In [ ]:
print("\n--- Starting Training ---")
FastLanguageModel.for_training(model)
trainer.train()

In [ ]:
print("\n--- Evaluating Fine-Tuned Model ---")
ft_metrics = evaluate_model(trainer.model, val_records, n=EVAL_N, label="Fine-Tuned Model")


In [ ]:
# =============================================================
# === 8. COMPARISON TABLE ===
# =============================================================
# Group metrics by category for readable output
import numpy as np
METRIC_GROUPS = {
    "Traditional (N-gram) — Baseline": [
        "rouge1", "rouge2", "rougeL", "rougeLsum"
    ],
    "Semantic (BERTScore on task_summary)": [
        "bertscore_precision", "bertscore_recall", "bertscore_f1"
    ],
    "Field-Level Exact Match (Atomic Accuracy)": [
        "exact_match_task_summary",
        "exact_match_assignee",
        "exact_match_issue_creation_date",
        "exact_match_all_fields",
    ],
    "Structured Output Quality": [
        "assignee_accuracy",
        "date_accuracy",
        "json_validity_rate",
    ],
    "Task-Specific (No-Action Detection)": [
        "no_action_precision",
        "no_action_recall",
        "no_action_f1",
    ],
}

print("\n" + "=" * 70)
print("  FULL EVALUATION COMPARISON: BASE vs FINE-TUNED")
print("=" * 70)

all_rows = []
for group_name, metric_keys in METRIC_GROUPS.items():
    print(f"\n  [{group_name}]")
    print(f"  {'Metric':<45} {'Base':>10} {'Fine-Tuned':>12} {'Delta':>10}")
    print(f"  {'-'*77}")
    for key in metric_keys:
        base_val = base_metrics.get(key, float("nan"))
        ft_val   = ft_metrics.get(key, float("nan"))
        delta    = ft_val - base_val if not (np.isnan(base_val) or np.isnan(ft_val)) else float("nan")
        delta_str = f"{delta:+.4f}" if not np.isnan(delta) else "   N/A"
        print(f"  {key:<45} {base_val:>10.4f} {ft_val:>12.4f} {delta_str:>10}")
        all_rows.append({
            "Group": group_name,
            "Metric": key,
            "Base Model": round(base_val, 4),
            "Fine-Tuned": round(ft_val, 4),
            "Delta": round(delta, 4) if not np.isnan(delta) else None,
        })

# Summary DataFrame
comparison_df = pd.DataFrame(all_rows)
print("\n\n" + "=" * 70)
print("  SUMMARY TABLE (pandas DataFrame)")
print("=" * 70)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)
print(comparison_df.to_string(index=False))

# Highlight key improvements
print("\n\n--- KEY IMPROVEMENTS ---")
key_metrics = [
    ("bertscore_f1",            "Semantic similarity of task summaries"),
    ("exact_match_all_fields",  "All 3 fields correct simultaneously"),
    ("assignee_accuracy",       "Correct assignee extracted"),
    ("no_action_f1",            "No-action detection F1"),
    ("json_validity_rate",      "Outputs valid JSON"),
]
for key, description in key_metrics:
    base_val = base_metrics.get(key, float("nan"))
    ft_val   = ft_metrics.get(key, float("nan"))
    if not (np.isnan(base_val) or np.isnan(ft_val)):
        delta = ft_val - base_val
        sign  = "+" if delta >= 0 else ""
        print(f"  {description:<45} {base_val:.4f} → {ft_val:.4f}  ({sign}{delta:.4f})")


In [ ]:
model.save_pretrained("base_model_qwen")
trainer.model.save_pretrained("finetuned_model_qwen")